# Compile responses and compute DAT

This notebook is adapted from Bellemare-Pepin (2026) DAT_GPT https://github.com/AntoineBellemare/DAT_GPT/


## Dependencies

In [ ]:
# NOTE: This notebook must be launched from the repository root for paths to work
import sys
import pandas as pd
import numpy as np
import json
import os
import re
import glob
import warnings
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats   
from pathlib import Path

warnings.filterwarnings('ignore')  # Suppress warnings for cleaner output

#REPO_ROOT = Path.cwd() # Assumes the notebook is launched from the repository root
REPO_ROOT = Path.cwd().parents[1]
sys.path.insert(0, str(REPO_ROOT))

# Import embedding models for computing DAT scores
from dat.embeddings import dat_glove, dat_sbert

DATA_PATH = REPO_ROOT / "data" / "dat" # raw data path
OUTPUT_PATH = REPO_ROOT

In [ ]:
# Initialize embedding models
sbert_model = dat_sbert.Model()
glove_model = dat_glove.Model(
    model=REPO_ROOT / "model" / "glove" / "glove.840B.300d.txt",
    dictionary=REPO_ROOT / "model" / "glove" / "words_glove.txt"
)

# Define a dictionary to store the results of model.dat(words)
results_dict = {'Temperature': [], 'Strategy': [], 'SBERT score': [], 'Glove score': [], 'Model': [], 'Words': [], 'Count': [], 'Valid count': []}

# Loop through each file in the data path
for file in sorted(glob.glob(str(DATA_PATH / "**" / "*.json"), recursive=True)):
    # Open the file and load the JSON data
    try:
        with open(file, 'r') as f:
            data = json.load(f)
            file = os.path.basename(file)

            # Loop through each key in the JSON data
            for i in data.keys():
                if data[i] is None:
                    print(f"Skipping key {i} because data is None")
                    continue
                
                words = data[i].split()
                print(i, "original words", words)

                # Find the indices of '1.' to '10.'
                indices = [m.start() for m in re.finditer(r'\b(?:[1-9]|10)\.', data[i])]  

                # Added: Stores the first word after <number>. and the rest in other_words 
                new_words = []
                other_words = []
                for idx in range(len(indices)):

                # Find the start of the segment (right after the <number>.)
                    match_str = re.match(r'\b(?:[1-9]|10)\.', data[i][indices[idx]:]).group()
                    start_idx = indices[idx] + len(match_str)

                    # Find the end of the segment (next index or end of string)
                    end_idx = indices[idx + 1] if idx < len(indices) - 1 else len(data[i])
                    segment = data[i][start_idx:end_idx].strip()
                    words = segment.split()
                    if words:
                        new_words.append(words[0])  # always store the first word

                        # Only add to other_words if this is not the segment after '10.'
                        if not (match_str.startswith('10.')) and len(words) > 1:
                            other_words.extend(words[1:])  # the rest

                # if no numbers are found, split by <number>., newline or comma. No bullet points
                if len(new_words) < 10:
                    pattern = r'\b(?:[1-9]|10)\.\s*|\n\s*[-*]\s*'
                    parts = re.split(pattern, data[i])

                    # Added: Combine new_words and split_data, remove duplicates while preserving order
                    split_data = [part.strip() for part in parts if part.strip()]
                
                    combined = new_words + split_data
                    seen = set()
                    new_words = []
                    for word in combined:
                        if word not in seen:
                            new_words.append(word)
                            seen.add(word)
                            
                if new_words != []:
                    words = new_words
                
                print(i, "combined words", words)

                # Split words that are comma-separated, newline-separated, or space-separated in a single string
                words = [word for sublist in words for word in re.split(r'[,\n\s]+', sublist) if word]
                
                print(i, "words after split", words)
                
                # Some lists contain an introductory statement to be removed
                if len(words) > 15: 

                    # Find any word containing a colon
                    try:
                        # Look for a word containing a colon and split on it
                        # Handle the case when the colon is part of a word
                        for i, word in enumerate(words):
                            if ':' in word:
                                words = words[i + 1:]
                                break # find the index of the colon
                    
                    except:
                        print("fail to removeintroductory statement", file, words)
                        continue

                # Remove "<nb>)" and "*" items from the list
                if len(words) > 15:
                    words = [word for word in words if not re.match(r'\d+\)', word) and not re.match(r'\*', word)]

                # Remove non-alphabetic characters from words (keep letters and hyphens)
                words = [re.sub(r'[^a-zA-Z-]', '', word) for word in words]

                # Remove letters following a single backslash, backslash with letters or numbers, or a new line
                words = [re.split(r'\\|\\.*|\n[0-9]*|\n\n', word)[0] for word in words]

                # Remove dobble asterisks from beginning and end of words for Gemini
                words = [re.sub(r'^\*+|\*+$', '', word) for word in words]

                # Remove "-" from the beginning of words
                words = [word.lstrip('-') for word in words]
                
                # Filter out words that aren't purely alphabetical while allowing "-" with at least one letter
                words = [word for word in words if all(c.isalpha() or c == '-' for c in word) and any(c.isalpha() for c in word)]

                # Remove empty strings
                words = [word for word in words if word.strip()]

                # lowercase the words
                words = [word.lower().strip() for word in words]
                
                words = words[:10]
                
                # Define the strategy based on the file name
                if 'dat' in file:
                    strategy = 'DAT'
                elif 'control' in file:
                    strategy = 'Control'

                # Define the temperature based on the file name
                if 'temp0.0' in file:
                    condition = 'Zero'
                elif 'temp1.5' in file:
                    condition = 'High'
                elif 'temp0.5' in file:
                    condition = 'Low'
                elif 'temp1.0' in file:
                    condition = 'Mid'

                # Define the model based on the file name
                if 'gpt-4-0314' in file:
                    llm = 'GPT-4-0314'
                elif 'gpt-4o-mini' in file: 
                    llm = 'GPT-4o-mini'
                elif 'gpt-3.5-turbo' in file:
                    llm = 'GPT-3.5-turbo'
                elif 'gpt-4-turbo' in file:
                    llm = 'GPT-4-turbo'
                elif 'gpt-4.5-preview-2025-02-27' in file:
                    llm = 'GPT-4.5-preview'
                elif 'gpt-4-0125-preview' in file:
                    llm = 'GPT-4-turbo-0125'
                elif 'gemini-1.5-pro' in file:
                    llm = 'GeminiPro1.5'
                elif 'gemini-2.5-pro-preview' in file:
                    llm = 'GeminiPro2.5'
                elif 'gemini-2.0-flash_' in file:
                    llm = 'GeminiFlash2.0'
                elif 'claude-3-5-sonnet' in file:
                    llm = 'Claude3.5-sonnet'
                elif 'claude-3-5-haiku' in file:
                    llm = 'Claude3.5-haiku'
                elif 'claude-3-opus' in file:
                    llm = 'Claude3-opus'
                elif 'claude-3-7-sonnet' in file:
                    llm = 'Claude3.7-sonnet'
                elif 'llama3-8B-instruct' in file:
                    llm = 'Llama3-8B-instruct'
                elif 'llama3-70B-instruct' in file:
                    llm = 'Llama3-70B-instruct'
                elif 'llama4-scout-instruct' in file:
                    llm = 'Llama4-scout-instruct'
            
                bert_score, num_valid = bert_model.dat(words)
                glove_score, _ = glove_model.dat(words)
                
                # Append the results to the dictionary
                results_dict['Temperature'].append(condition)
                results_dict['Strategy'].append(strategy)
                results_dict['Score'].append(bert_score)
                results_dict['Glove score'].append(glove_score)
                results_dict['Words'].append(words)
                results_dict['Model'].append(llm)
                results_dict['Count'].append(len(words))
                results_dict['Valid count'].append(num_valid)

    except json.JSONDecodeError as e:
        print(f"Error reading file {file}: {e}")
        continue

# Convert the results dictionary to a Pandas DataFrame
results_df = pd.DataFrame(results_dict)

# Save to CSV file
results_df.to_csv(str(OUTPUT_PATH), index=False)